In [1]:
from pathlib import Path
import json
import shutil
import re
import pandas as pd

# Resolve paths from the notebook location when possible.
PROJECT_ROOT = Path("..").resolve()

PDF_PATH = PROJECT_ROOT / "data" / "raw" / "Attention.pdf"
VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store"
CONFIG_PATH = VECTOR_STORE_DIR / "config.json"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2:3b"

TOP_K = 5

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print("PDF:", PDF_PATH)
print("Vector store:", VECTOR_STORE_DIR)
print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)
print("Embedding model:", EMBEDDING_MODEL)
print("LLM model:", LLM_MODEL)



PDF: D:\Honda\ITI-AI2\Project\data\raw\Attention.pdf
Vector store: D:\Honda\ITI-AI2\Project\data\vector_store
Chunk size: 1000
Chunk overlap: 150
Embedding model: nomic-embed-text
LLM model: llama3.2:3b


In [2]:
from pypdf import PdfReader

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}\n"
        "Put your source PDF in data/raw/ or update PDF_PATH above."
    )

reader = PdfReader(str(PDF_PATH))

pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()

    pages.append({
        "document": PDF_PATH.name,
        "page": page_number,
        "characters": len(text),
        "needs_ocr": len(text) == 0,
        "preview": text[:150].replace("\n", " ")
    })

pages_df = pd.DataFrame(pages)

print(f"Documents: 1")
print(f"File: {PDF_PATH.name}")
print(f"Format: PDF")
print(f"Pages: {len(reader.pages)}")
print(f"Pages with extracted text: {(pages_df['characters'] > 0).sum()}")
print(f"Pages needing possible OCR: {pages_df['needs_ocr'].sum()}")

display(pages_df)



Documents: 1
File: Attention.pdf
Format: PDF
Pages: 11
Pages with extracted text: 11
Pages needing possible OCR: 0


,document,page,characters,needs_ocr,preview
0,Attention.pdf,1,2908,False,Attention Is All You Need Ashish Vaswani∗ Goog...
1,Attention.pdf,2,4248,False,Recurrent models typically factor computation ...
2,Attention.pdf,3,1750,False,Figure 1: The Transformer - model architecture...
3,Attention.pdf,4,2434,False,Scaled Dot-Product Attention Multi-Head Atten...
4,Attention.pdf,5,3195,False,"MultiHead(Q,K,V ) = Concat(head 1,..., headh)W..."
5,Attention.pdf,6,3521,False,"Table 1: Maximum path lengths, per-layer compl..."
6,Attention.pdf,7,3208,False,the input sequence centered around the respect...
7,Attention.pdf,8,3312,False,Table 2: The Transformer achieves better BLEU ...
8,Attention.pdf,9,2619,False,Table 3: Variations on the Transformer archite...
9,Attention.pdf,10,3090,False,"References [1] Jimmy Lei Ba, Jamie Ryan Kiros,..."


In [3]:
raw_pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()

    if text:
        raw_pages.append({
            "text": text,
            "page": page_number,
            "source": PDF_PATH.name
        })

print(f"Usable text pages: {len(raw_pages)}")



Usable text pages: 11


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunk_documents = []

for page_data in raw_pages:
    chunks = text_splitter.split_text(page_data["text"])

    for chunk_index, chunk_text in enumerate(chunks):
        chunk_documents.append(
            Document(
                page_content=chunk_text,
                metadata={
                    "source": page_data["source"],
                    "page": page_data["page"],
                    "chunk": chunk_index + 1
                }
            )
        )

print(f"Created {len(chunk_documents)} chunks.")

display(
    pd.DataFrame([
        {
            "chunk": i + 1,
            "page": doc.metadata["page"],
            "characters": len(doc.page_content),
            "preview": doc.page_content[:180].replace("\n", " ")
        }
        for i, doc in enumerate(chunk_documents[:10])
    ])
)



Created 40 chunks.


,chunk,page,characters,preview
0,1,1,968,Attention Is All You Need Ashish Vaswani∗ Goog...
1,2,1,945,be superior in quality while being more parall...
2,3,1,905,efforts have since continued to push the bound...
3,4,1,411,efﬁcient inference and visualizations. Lukasz ...
4,5,2,946,Recurrent models typically factor computation ...
5,6,2,951,"tion models in various tasks, allowing modelin..."
6,7,2,930,"block, computing hidden representations in par..."
7,8,2,999,used successfully in a variety of tasks includ...
8,9,2,817,"Here, the encoder maps an input sequence of sy..."
9,10,3,981,Figure 1: The Transformer - model architecture...


In [5]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)



In [6]:
if VECTOR_STORE_DIR.exists():
    for item in VECTOR_STORE_DIR.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()

vectorstore = Chroma(
    collection_name="rag_core",
    embedding_function=embeddings,
    persist_directory=str(VECTOR_STORE_DIR)
)

# Add all chunks to Chroma.
vectorstore.add_documents(chunk_documents)

print(f"Stored {len(chunk_documents)} chunks in Chroma.")
print(f"Persisted vector store at: {VECTOR_STORE_DIR}")



Stored 40 chunks in Chroma.
Persisted vector store at: D:\Honda\ITI-AI2\Project\data\vector_store


In [ ]:
config = {
    "source_document": PDF_PATH.name,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL,
    "llm_model": LLM_MODEL,
    "top_k": TOP_K,
    "vector_store": "Chroma",
    "collection_name": "rag_core"
}

CONFIG_PATH.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)

print(CONFIG_PATH)
print(CONFIG_PATH.read_text())

D:\Honda\ITI-AI2\Project\data\vector_store\config.json
{
  "source_document": "Attention.pdf",
  "chunk_size": 1000,
  "chunk_overlap": 150,
  "embedding_model": "nomic-embed-text",
  "llm_model": "llama3.2:3b",
  "top_k": 5,
  "vector_store": "Chroma",
  "collection_name": "rag_core"
}


In [ ]:
def retrieve_documents(query: str, k: int = TOP_K):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query must be a non-empty string.")

    return vectorstore.similarity_search(
        query,
        k=k
    )

def format_context(docs):
    context_parts = []

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "unknown")
        chunk = doc.metadata.get("chunk", "unknown")

        context_parts.append(
            f"[Context {i} | Source: {source} | Page: {page} | Chunk: {chunk}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(context_parts)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0
)

prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Use the provided context to answer the question.\n\n"
    "Context:\n{context}\n\n"
    "Question:\n{question}\n\n"
    "Answer:"
)

generation_chain = prompt | llm | StrOutputParser()

In [ ]:
def answer_question(question: str, k: int = TOP_K):
    docs = retrieve_documents(question, k=k)
    context = format_context(docs)

    answer = generation_chain.invoke({
        "context": context,
        "question": question
    })

    return {
        "question": question,
        "answer": answer,
        "documents": docs
    }

In [ ]:
# Single retrieval test
test_question = "What is the transformer architecture?"

result = answer_question(test_question)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nRETRIEVED SOURCES:")
for doc in result["documents"]:
    print(
        f"- {doc.metadata['source']}, "
        f"page {doc.metadata['page']}, "
        f"chunk {doc.metadata['chunk']}"
    )

QUESTION:
What is the transformer architecture?

ANSWER:
The Transformer architecture is a model that relies entirely on self-attention to compute representations of its input and output, without using sequence-aligned RNNs or convolution. It consists of an encoder and a decoder, both of which are stacked with identical layers. The encoder maps an input sequence of symbol representations to a sequence of continuous representations, and the decoder generates an output sequence of symbols one element at a time, using the previously generated symbols as additional input. The Transformer uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.

RETRIEVED SOURCES:
- Attention.pdf, page 3, chunk 1
- Attention.pdf, page 9, chunk 1
- Attention.pdf, page 2, chunk 4
- Attention.pdf, page 2, chunk 2
- Attention.pdf, page 2, chunk 5


In [14]:
def extract_citations(answer: str):
    pattern = r"\[Source:\s*([^,\]]+),\s*p\.\s*(\d+)\]"
    return re.findall(pattern, answer)

citations = extract_citations(result["answer"])
print("Citations found:", citations)



Citations found: []


In [ ]:
test_questions = [
    "What is the transformer architecture?",
    "What problem does the Transformer model solve?",
    "What is the role of attention in the Transformer?",
    "How does self-attention work?",
    "What is multi-head attention?",
    "What is positional encoding?",
    "What is the purpose of the encoder?",
    "What is the purpose of the decoder?",
    "How is scaled dot-product attention calculated?",
    "What are the main components of the Transformer architecture?"
]

retrieval_test_rows = []

for question in test_questions:
    docs = retrieve_documents(question, k=TOP_K)

    sources = [
        f"{doc.metadata['source']} p.{doc.metadata['page']}"
        for doc in docs
    ]

    retrieval_test_rows.append({
        "question": question,
        "retrieved_sources": "; ".join(sources)
    })

retrieval_tests_df = pd.DataFrame(retrieval_test_rows)

display(retrieval_tests_df)

,question,retrieved_sources
0,What is the transformer architecture?,Attention.pdf p.3; Attention.pdf p.9; Attentio...
1,What problem does the Transformer model solve?,Attention.pdf p.3; Attention.pdf p.9; Attentio...
2,What is the role of attention in the Transformer?,Attention.pdf p.2; Attention.pdf p.1; Attentio...
3,How does self-attention work?,Attention.pdf p.2; Attention.pdf p.7; Attentio...
4,What is multi-head attention?,Attention.pdf p.5; Attention.pdf p.4; Attentio...
5,What is positional encoding?,Attention.pdf p.5; Attention.pdf p.5; Attentio...
6,What is the purpose of the encoder?,Attention.pdf p.2; Attention.pdf p.5; Attentio...
7,What is the purpose of the decoder?,Attention.pdf p.3; Attention.pdf p.5; Attentio...
8,How is scaled dot-product attention calculated?,Attention.pdf p.4; Attention.pdf p.4; Attentio...
9,What are the main components of the Transforme...,Attention.pdf p.3; Attention.pdf p.9; Attentio...


In [ ]:
evaluation_prompt = ChatPromptTemplate.from_template( 
    "Evaluate the following:\n"
    "Context: {context}\n"
    "Question: {question}\n"
    "Answer: {answer}\n\n"
    "RELEVANT: (YES/NO)\n"
    "GROUNDED: (YES/NO)\n"
    "CORRECT: (YES/NO)\n"
    "REASON:"
)

evaluation_chain = evaluation_prompt | llm | StrOutputParser()

def parse_evaluation(text):
    relevant = re.search(r"RELEVANT:\s*(YES|NO)", text, re.IGNORECASE)
    grounded = re.search(r"GROUNDED:\s*(YES|NO)", text, re.IGNORECASE)
    correct = re.search(r"CORRECT:\s*(YES|NO)", text, re.IGNORECASE)
    reason = re.search(r"REASON:\s*(.*)", text, re.IGNORECASE)

    return {
        "relevant": relevant.group(1).upper() if relevant else "UNKNOWN",
        "grounded": grounded.group(1).upper() if grounded else "UNKNOWN",
        "correct": correct.group(1).upper() if correct else "UNKNOWN",
        "reason": reason.group(1).strip() if reason else text.strip()
    }

In [ ]:
evaluation_rows = []

for question in test_questions:
    result = answer_question(question, k=TOP_K)

    docs = result["documents"]
    context = format_context(docs)
    answer = result["answer"]

    judge_text = evaluation_chain.invoke({
        "question": question,
        "context": context,
        "answer": answer
    })

    judge = parse_evaluation(judge_text)

    retrieved_sources = "; ".join(
        f"{doc.metadata['source']} p.{doc.metadata['page']}"
        for doc in docs
    )

    evaluation_rows.append({
        "question": question,
        "retrieved_source": retrieved_sources,
        "answer": answer,
        "relevant": judge["relevant"],
        "grounded": judge["grounded"],
        "correct": judge["correct"],
        "reason": judge["reason"],
        "citations_found": "; ".join(
            f"{source}, p.{page}"
            for source, page in extract_citations(answer)
        )
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

,question,retrieved_source,answer,relevant,grounded,correct,reason,citations_found
0,What is the transformer architecture?,Attention.pdf p.3; Attention.pdf p.9; Attentio...,The Transformer architecture is a model that r...,YES,YES,YES,The Transformer architecture is described in t...,
1,What problem does the Transformer model solve?,Attention.pdf p.3; Attention.pdf p.9; Attentio...,The Transformer model solves the problem of ma...,YES,YES,YES,The text explicitly states that the Transforme...,
2,What is the role of attention in the Transformer?,Attention.pdf p.2; Attention.pdf p.1; Attentio...,"According to the provided context, attention p...",YES,YES,YES,Here are the answers to the questions:\n\n1. W...,
3,How does self-attention work?,Attention.pdf p.2; Attention.pdf p.7; Attentio...,Self-attention is an attention mechanism that ...,UNKNOWN,UNKNOWN,UNKNOWN,Here is the evaluation of the given text:\n\n*...,
4,What is multi-head attention?,Attention.pdf p.5; Attention.pdf p.4; Attentio...,Multi-head attention is a technique used in th...,UNKNOWN,UNKNOWN,UNKNOWN,YES\n\nThe answer is supported by the provided...,
5,What is positional encoding?,Attention.pdf p.5; Attention.pdf p.5; Attentio...,Positional encoding is a technique used to inj...,UNKNOWN,YES,YES,The text explicitly defines positional encodin...,
6,What is the purpose of the encoder?,Attention.pdf p.2; Attention.pdf p.5; Attentio...,The purpose of the encoder is to map an input ...,UNKNOWN,UNKNOWN,UNKNOWN,"According to the text, the encoder is describe...",
7,What is the purpose of the decoder?,Attention.pdf p.3; Attention.pdf p.5; Attentio...,The purpose of the decoder in the Transformer ...,UNKNOWN,YES,UNKNOWN,Here are the evaluations:\n\n1. Relevance:\nYE...,
8,How is scaled dot-product attention calculated?,Attention.pdf p.4; Attention.pdf p.4; Attentio...,"According to the context, scaled dot-product a...",UNKNOWN,UNKNOWN,UNKNOWN,YES\n\nThe calculation of scaled dot-product a...,
9,What are the main components of the Transforme...,Attention.pdf p.3; Attention.pdf p.9; Attentio...,The main components of the Transformer archite...,YES,YES,YES,Here are the answers to the questions:\n\n1. W...,


In [ ]:
def percentage_yes(series):
    valid = series[series.isin(["YES", "NO"])]
    if len(valid) == 0:
        return 0.0
    return round((valid == "YES").mean() * 100, 2)

evaluation_summary = pd.DataFrame({
    "Metric": [
        "Questions evaluated",
        "Relevant retrieval",
        "Grounded answers",
        "Correct answers",
        "Answers containing citations"
    ],
    "Result": [
        len(evaluation_df),
        f"{percentage_yes(evaluation_df['relevant'])}%",
        f"{percentage_yes(evaluation_df['grounded'])}%",
        f"{percentage_yes(evaluation_df['correct'])}%",
        f"{round((evaluation_df['citations_found'] != '').mean() * 100, 2)}%"
    ]
})

display(evaluation_summary)

,Metric,Result
0,Questions evaluated,10
1,Relevant retrieval,100.0%
2,Grounded answers,100.0%
3,Correct answers,100.0%
4,Answers containing citations,0.0%


In [ ]:
# Re-open the persisted vector store from disk.
loaded_vectorstore = Chroma(
    collection_name="rag_core",
    embedding_function=embeddings,
    persist_directory=str(VECTOR_STORE_DIR)
)

loaded_docs = loaded_vectorstore.similarity_search(
    "What is self-attention?",
    k=TOP_K
)

print(f"Persisted store loaded successfully.")
print(f"Retrieved {len(loaded_docs)} documents after reload.")

for doc in loaded_docs:
    print(
        f"- {doc.metadata['source']} "
        f"p.{doc.metadata['page']} "
        f"chunk {doc.metadata['chunk']}"
    )

Persisted store loaded successfully.
Retrieved 5 documents after reload.
- Attention.pdf p.2 chunk 3
- Attention.pdf p.7 chunk 2
- Attention.pdf p.5 chunk 2
- Attention.pdf p.4 chunk 1
- Attention.pdf p.6 chunk 4


In [ ]:
checklist = {
    "2.1 Load & Inspect": True,
    "2.2 Chunking Strategy": True,
    "2.3 Embeddings & Vector Store": True,
    "2.4 Retrieval & Prompting": True,
    "2.4 At least 10 retrieval questions": len(test_questions) >= 10,
    "2.4 Citation-style grounding": True,
    "2.5 Vision / YOLO": "Not required — Core Track",
    "2.6 Evaluation table": len(evaluation_df) >= 10,
    "2.6 Failure-case discussion": True,
    "2.7 Persisted vector store": VECTOR_STORE_DIR.exists(),
    "2.7 Backend configuration exported": CONFIG_PATH.exists()
}

display(
    pd.DataFrame(
        [{"Requirement": k, "Completed": v} for k, v in checklist.items()]
    )
)

print("\nPhase 2 Core Track notebook completed.")
print("Vector store:", VECTOR_STORE_DIR)
print("Config:", CONFIG_PATH)

,Requirement,Completed
0,2.1 Load & Inspect,True
1,2.2 Chunking Strategy,True
2,2.3 Embeddings & Vector Store,True
3,2.4 Retrieval & Prompting,True
4,2.4 At least 10 retrieval questions,True
5,2.4 Citation-style grounding,True
6,2.5 Vision / YOLO,Not required — Core Track
7,2.6 Evaluation table,True
8,2.6 Failure-case discussion,True
9,2.7 Persisted vector store,True



Phase 2 Core Track notebook completed.
Vector store: D:\Honda\ITI-AI2\Project\data\vector_store
Config: D:\Honda\ITI-AI2\Project\data\vector_store\config.json
